In [37]:
import sys
import subprocess
import os

# --- 1. 尝试导入 (解决名字混淆问题) ---
print("🚀 正在尝试加载 Pinocchio 库...")

try:
    # 核心修正：虽然包名叫 'pin'，但导入时必须用 'pinocchio'
    import pinocchio as pin
    print(f"🎉 成功导入 'pinocchio' (别名 pin) ! 版本: {pin.__version__}")
    
except ImportError:
    print("⚠️ 'import pinocchio' 失败，尝试检查安装路径...")
    
    # 如果导入失败，可能是路径没加进去，手动加一下
    # 根据你的日志，库在 /usr/local/lib/python3.10/dist-packages
    global_path = "/usr/local/lib/python3.10/dist-packages"
    if global_path not in sys.path:
        sys.path.append(global_path)
        print(f"已手动添加搜索路径: {global_path}")
    
    try:
        import pinocchio as pin
        print(f"🎉 重试后成功导入! 版本: {pin.__version__}")
    except ImportError as e:
        print(f"❌ 依然失败. 错误信息: {e}")
        print("请尝试在下方的终端(Terminal)里运行: sudo chmod -R 755 /usr/local/lib/python3.10/dist-packages")

# --- 2. 补全其他画图库 ---
def install(package):
    try:
        __import__(package)
    except ImportError:
        print(f"📦 正在补充安装: {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, 
                               "-i", "https://pypi.tuna.tsinghua.edu.cn/simple", "--user"])

install("plotly")
install("scipy")
install("pandas")
install("matplotlib")

# --- 3. 最终环境确认 ---
import numpy as np
import plotly.graph_objects as go

print("-" * 30)
if 'pin' in locals():
    print("✅ 环境完美！后续代码请直接使用 'pin' 来调用函数。")
    print(f"例如: model = pin.buildModelFromUrdf(...)")
else:
    print("❌ 环境依然有问题，请联系指导老师或检查 Docker 权限。")

🚀 正在尝试加载 Pinocchio 库...
🎉 成功导入 'pinocchio' (别名 pin) ! 版本: 3.8.0
------------------------------
✅ 环境完美！后续代码请直接使用 'pin' 来调用函数。
例如: model = pin.buildModelFromUrdf(...)


In [38]:
# --- 配置区域 ---

# 1. 你的 URDF 文件路径
urdf_path = "/anyverse/sub_modules/isaacros/src/wheeled_humanoid/robot_model/urdf/wheel_robot.urdf"

# 2. 是否强制使用虚拟模型 (调试用)
USE_DUMMY_ROBOT = False 

# ----------------

print(f"🔍 正在寻找模型文件: {urdf_path}")

if USE_DUMMY_ROBOT or not os.path.exists(urdf_path):
    if not USE_DUMMY_ROBOT:
        print(f"⚠️ 警告: 找不到路径 {urdf_path}")
    print("🛠️ 将构建一个【虚拟 7轴机械臂】用于演示流程...")
    
    # 使用 pin.buildSampleModelManipulator()
    model = pin.buildSampleModelManipulator() 
    
    # 这里的 model.nq 是自由度，如果生成的不是7轴，我们可以不管，先跑通流程
else:
    print(f"✅ 找到文件，正在加载...")
    # 关键修改：使用 pin.buildModelFromUrdf (不再是 pinocchio.)
    model = pin.buildModelFromUrdf(urdf_path)

# 创建数据对象 (用于存储计算过程中的速度、加速度、位置结果)
data = model.createData()

print("-" * 30)
print(f"✅ 模型加载成功！")
print(f"📊 关节数量 (njoints): {model.njoints} (包含基座)")
print(f"🔧 自由度 (nq): {model.nq}")

# 如果自由度 > 7 (比如双臂)，我们后续代码需要适配
if model.nq > 7:
    print("💡 提示：检测到这是一个多自由度机器人(如双臂)，后续采样代码可能需要稍作调整。")

# --- 调试代码：查找真实的 Frame 名字 ---
print(f"当前模型总共有 {model.nframes} 个 Frame。")

# 打印所有名字包含 'left' 或 'gripper' 的 Frame，帮你缩小范围
print("\n=== 可能是左手的 Frame ===")
for frame in model.frames:
    if "left" in frame.name or "L_" in frame.name: # 根据命名习惯过滤
        print(f"ID: {model.getFrameId(frame.name)} | Name: {frame.name}")

print("\n=== 可能是末端(gripper)的 Frame ===")
for frame in model.frames:
    if "gripper" in frame.name or "hand" in frame.name or "tip" in frame.name:
        print(f"ID: {model.getFrameId(frame.name)} | Name: {frame.name}")

🔍 正在寻找模型文件: /anyverse/sub_modules/isaacros/src/wheeled_humanoid/robot_model/urdf/wheel_robot.urdf
✅ 找到文件，正在加载...
------------------------------
✅ 模型加载成功！
📊 关节数量 (njoints): 33 (包含基座)
🔧 自由度 (nq): 32
💡 提示：检测到这是一个多自由度机器人(如双臂)，后续采样代码可能需要稍作调整。
当前模型总共有 108 个 Frame。

=== 可能是左手的 Frame ===
ID: 4 | Name: chassis_left_Joint
ID: 5 | Name: chassis_left_Link
ID: 32 | Name: left_fixed
ID: 33 | Name: AR5_5_07L_base
ID: 34 | Name: AR5_5_07L_joint_1
ID: 35 | Name: AR5_5_07L_link1
ID: 36 | Name: AR5_5_07L_joint_2
ID: 37 | Name: AR5_5_07L_link2
ID: 38 | Name: AR5_5_07L_joint_3
ID: 39 | Name: AR5_5_07L_link3
ID: 40 | Name: AR5_5_07L_joint_4
ID: 41 | Name: AR5_5_07L_link4
ID: 42 | Name: AR5_5_07L_joint_5
ID: 43 | Name: AR5_5_07L_link5
ID: 44 | Name: AR5_5_07L_joint_6
ID: 45 | Name: AR5_5_07L_link6
ID: 46 | Name: AR5_5_07L_joint_7
ID: 47 | Name: AR5_5_07L_link7
ID: 48 | Name: AR5_5_07L_tcp_joint
ID: 49 | Name: AR5_5_07L_tcp
ID: 50 | Name: left_flange_base_joint
ID: 51 | Name: left_flange_link
ID: 52 | Name: l

In [39]:
def get_chain_q_indices(model, frame_name, stop_at_root=True):
    """
    自动回溯运动链，找到该 Frame 所属支路的所有关节在 q 中的索引。
    
    原理：
    从末端 Frame 开始，沿着 parent 往上爬。
    记录路径上所有的关节 ID。
    """
    if not model.existFrame(frame_name):
        raise ValueError(f"Frame '{frame_name}' 不存在！")
        
    frame_id = model.getFrameId(frame_name)
    joint_id = model.frames[frame_id].parent
    
    chain_joint_ids = []
    
    # 开始回溯
    while joint_id > 0: # 0 是 universe (世界坐标系)，不需要
        # 如果是 FreeFlyer (浮动基座)，通常我们要把它排除，或者根据需求保留
        # 这里我们假设遇到 FreeFlyer 就停止，或者一直追溯到根
        if model.joints[joint_id].shortname() == "JointModelFreeFlyer":
            if stop_at_root:
                break
        
        chain_joint_ids.append(joint_id)
        joint_id = model.parents[joint_id]
        
    # 将关节 ID 转换为 q 向量中的索引 (idx_q)
    # 注意：一个关节可能有多个 q (比如球关节)，也可能没有 q (固定关节)
    q_indices = []
    for jid in chain_joint_ids:
        idx_q = model.joints[jid].idx_q
        nq = model.joints[jid].nq
        if idx_q >= 0 and nq > 0:
            # 把该关节所有的 q 维度都加进去
            q_indices.extend(list(range(idx_q, idx_q + nq)))
            
    return set(q_indices)

def get_dual_arm_indices(model, left_name, right_name):
    """
    计算【纯手臂】的关节索引（排除腰部、头、基座）。
    
    原理：
    1. 找到左手全路径 (含腰)
    2. 找到右手全路径 (含腰)
    3. 取【对称差集】：(左 U 右) - (左 n 右)
       这样会自动剔除掉两只手共用的父节点（即躯干、腰、基座）
    """
    q_idx_left = get_chain_q_indices(model, left_name)
    q_idx_right = get_chain_q_indices(model, right_name)
    
    # 纯手臂 = (左手路径) 与 (右手路径) 的不重合部分
    # 这样腰部(Waist)因为是公共父节点，会被自动剔除！
    pure_arm_indices = list(q_idx_left.symmetric_difference(q_idx_right))
    
    return pure_arm_indices

import time
from tqdm import tqdm

# --- 1. 手动指定左右手的名字 (请根据你的 URDF 修改!) ---
# 根据你之前的日志，你的末端名字里应该包含 gripper
# 请务必核对你的模型，这里假设是 left_gripper_base_joint 和 right_gripper_base_joint
name_left = "left_gripper_base_joint"   # <--- 请核对
name_right = "right_gripper_base_joint" # <--- 请核对

print(f"🎯 双臂分析模式:")
print(f"   左手: {name_left}")
print(f"   右手: {name_right}")

try:
    id_left = model.getFrameId(name_left)
    id_right = model.getFrameId(name_right)
except:
    print("❌ 错误：找不到指定的 Frame 名字，请检查 print(model.frames) 的输出")
    raise ValueError("Frame name not found")

# --- 1. 自动分析拓扑结构 ---
print("🧩 正在自动分析机器人拓扑结构...")

# 只要你给了末端名字，剩下的全自动
arm_q_indices = get_dual_arm_indices(model, name_left, name_right)

print(f"   自动锁定纯手臂关节 q 索引: {arm_q_indices}")
print(f"   (已自动排除腰部、头部和移动底盘)")

# --- 2. 采样设置 ---
num_samples = 50000
points = []       
manipulability = [] 

print(f"\n🚀 开始双臂采样 ({num_samples} 次)...")
start_time = time.time()

q_neutral = pin.neutral(model) # 获取标准站立姿态

points_L = []
points_R = []

for i in tqdm(range(num_samples), desc="通用采样"):
    # A. 生成全随机姿态
    q_rand = pin.randomConfiguration(model)
    
    # B. 【通用混合策略】
    # 先复制一份完全静止的站立姿态
    q = q_neutral.copy()
    
    # 只把【纯手臂】部分的数值，替换成随机生成的数值
    # 其他部分（基座、腰、头）全部保持 neutral 不动！
    for idx in arm_q_indices:
        q[idx] = q_rand[idx]

    # FK
    pin.forwardKinematics(model, data, q)
    pin.updateFramePlacements(model, data)
    
    # --- 左手数据 ---
    pos_L = data.oMf[id_left].translation.copy()
    pin.computeJointJacobians(model, data, q)
    J_L = pin.getFrameJacobian(model, data, id_left, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)[:3, :]
    w_L = np.sqrt(np.linalg.det(J_L @ J_L.T))
    
    # --- 右手数据 ---
    pos_R = data.oMf[id_right].translation.copy()
    # 注意：同一个 q 下，雅可比需要重新获取 frame 对应的
    J_R = pin.getFrameJacobian(model, data, id_right, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)[:3, :]
    w_R = np.sqrt(np.linalg.det(J_R @ J_R.T))

    # 同时存入列表
    points.append(pos_L)
    manipulability.append(w_L)
    
    points.append(pos_R)
    manipulability.append(w_R)

# 转换格式
points = np.array(points)
manipulability = np.array(manipulability)

print("-" * 30)
print(f"✅ 计算完成！共生成 {len(points)} 个点 (左手+右手)")

/tmp/ipykernel_4084926/3143261657.py:13: UserWarning:

Deprecated member. Use Frame.parentJoint instead.



🎯 双臂分析模式:
   左手: left_gripper_base_joint
   右手: right_gripper_base_joint
🧩 正在自动分析机器人拓扑结构...
   自动锁定纯手臂关节 q 索引: [6, 7, 8, 9, 10, 11, 12, 19, 20, 21, 22, 23, 24, 25]
   (已自动排除腰部、头部和移动底盘)

🚀 开始双臂采样 (50000 次)...


通用采样: 100%|██████████| 50000/50000 [00:00<00:00, 50839.57it/s]

------------------------------
✅ 计算完成！共生成 100000 个点 (左手+右手)


In [40]:
import sys
import subprocess
import plotly.graph_objects as go
import numpy as np

# --- 0. 自动修复环境 ---
try:
    import nbformat
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat>=4.2.0", "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"])

# --- 1. 确定标题名称 (兼容单臂和双臂模式) ---
# 检查是否存在双臂变量，如果不存在，就尝试用旧的单臂变量，实在没有就写默认值
if 'name_left' in locals() and 'name_right' in locals():
    title_text = f"双臂可达性分析<br><sub>Left: {name_left} | Right: {name_right}</sub>"
elif 'target_frame_name' in locals():
    title_text = f"单臂可达性分析<br><sub>Target: {target_frame_name}</sub>"
else:
    title_text = "机械臂可达性分析 (末端未知)"

# --- 2. 安全检查 ---
if 'points' not in locals() or len(points) == 0:
    print("❌ 错误：找不到点云数据！请先运行代码块 3。")
else:
    print(f"🎨 正在生成 3D 热力图 (共 {len(points)} 个采样点)...")
    
    # --- 3. 创建散点图 ---
    scatter = go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(
            size=2,                
            color=manipulability,  
            colorscale='Turbo',    # 推荐用 Turbo 或 Jet，对比度高
            opacity=0.3,           
            colorbar=dict(title="灵活性指数 (Manipulability)")
        ),
        name='可达工作空间'
    )

    # --- 4. 标记基座原点 ---
    base_marker = go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=5, color='black', symbol='x'),
        name='基座 (Base)'
    )

    # --- 5. 布局设置 ---
    fig = go.Figure(data=[scatter, base_marker])

    fig.update_layout(
        title=title_text, # <--- 这里修复了！
        scene=dict(
            xaxis_title='X (前向) [m]',
            yaxis_title='Y (横向) [m]',
            zaxis_title='Z (高度) [m]',
            aspectmode='data' # 强制比例一致，这对于看只有一侧的非对称空间很重要
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        height=700
    )

    # --- 6. 显示与保存 ---
    try:
        fig.show()
    except Exception:
        pass
    
    # 无论显示是否成功，都保存一份网页，双重保险
    save_name = "dual_arm_reachability.html"
    fig.write_html(save_name)
    print(f"✅ 图表已保存为 '{save_name}'，请在左侧文件列表下载或打开查看。")

🎨 正在生成 3D 热力图 (共 100000 个采样点)...


✅ 图表已保存为 'dual_arm_reachability.html'，请在左侧文件列表下载或打开查看。


In [41]:
import sys
import subprocess
import os
import numpy as np
import pinocchio as pin # 确保 pin 已导入

# --- 1. 自动安装 meshcat ---
try:
    import meshcat
    import meshcat.geometry as g
except ImportError:
    print("🔧 安装 meshcat...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "meshcat", 
                           "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"])
    import meshcat
    import meshcat.geometry as g

from pinocchio.visualize import MeshcatVisualizer

# --- 2. 准备几何模型 (修复路径问题) ---
# 这里的逻辑是：URDF 里的 package://robot_model/ 对应的是硬盘上的 .../wheeled_humanoid/robot_model/
# 所以我们需要把 .../wheeled_humanoid/ 这个路径加到搜索列表里

dir_urdf = os.path.dirname(urdf_path) # .../urdf
dir_pkg = os.path.dirname(dir_urdf)   # .../robot_model
dir_root = os.path.dirname(dir_pkg)   # .../wheeled_humanoid <--- 关键！

# 构建搜索路径列表
mesh_dirs = [dir_root, dir_pkg, dir_urdf, "/anyverse"]

print("🎨 正在加载机器人 3D 模型...")

try:
    # 重新构建模型，这次带上几何搜索路径
    model, collision_model, visual_model = pin.buildModelsFromUrdf(
        urdf_path, 
        mesh_dirs, 
        pin.JointModelFreeFlyer() if model.nq > 7 else None
    )
    
    # 初始化 Visualizer
    viz = MeshcatVisualizer(model, collision_model, visual_model)
    
    # 强制开启 MeshCat 服务
    # url_type="tcp" 确保绑定到本地端口
    viz.initViewer(open=False) 
    viz.loadViewerModel()
    
    print("✅ 机器人模型加载成功！(不再是圆柱体了)")

except Exception as e:
    print(f"❌ 模型加载失败: {e}")
    print("⚠️ 依然使用简易模式...")
    viz = None

# --- 3. 渲染点云 ---
if viz is not None:
    viewer = viz.viewer
    
    # 颜色映射
    if len(manipulability) > 0:
        colors = np.zeros((3, len(points)))
        max_m = np.max(manipulability)
        colors[0, :] = manipulability / max_m # R
        colors[2, :] = 1.0 - (colors[0, :])   # B
    else:
        colors = np.array([1, 0, 0])

    # 添加点云
    viewer["reachability_cloud"].set_object(
        g.PointCloud(position=points.T, color=colors, size=0.01)
    )
    
    # 显示零位姿态
    viz.display(pin.neutral(model))

    print("-" * 30)
    print("🌐 可视化服务已启动，请进行端口转发操作！")

🎨 正在加载机器人 3D 模型...
You can open the visualizer by visiting the following URL:
http://127.0.0.1:7014/static/
✅ 机器人模型加载成功！(不再是圆柱体了)
------------------------------
🌐 可视化服务已启动，请进行端口转发操作！
